# Baseline Tasks

> This section provides an overview of the baseline tasks for benchmarking nanobot capabilities.

In [ ]:
#| default_exp baseline

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from datasets import load_dataset
from pathlib import Path
from pydantic import BaseModel, Field
from typing import List, Literal, Any
import json

## Baseline Task Template

For our experiment setup, we will use a fixed template for all baseline tasks. The template declares the necessary metadata for tasks, including the fields as following:

```json
{
  "id": "gaia_001",
  "source": "GAIA",
  "source_split": "level_1",
  "task_family": "research_synthesis",
  "difficulty": "medium",
  "prompt": "the original user request",

  "primary_kpi": [
    "Reasoning-First Success Rate",
    "Decomposition Stability"
  ],

  "tool_count_class": "multi_tool",
  "composition_depth": "2_3",
  "planning_horizon": "medium",
  "cross_api_mashup": true,

  "available_tools_label": [
    "web_search",
    "read_file",
    "write_file"
  ],

  "goal_condition": {
    "type": "text_match",
    "target": "expected answer string",
    "match_mode": "exact"
  },

  "judge_mode": "automatic",
  "risk_tags": [
    "multi_step",
    "tool_choice",
    "context_fragility"
  ],

  "intervention_policy": {
    "reasoning_first": true,
    "soft_scaffold_allowed": ["prompt_scaffold", "memory", "combined"],
    "hard_fallback_allowed": false
  }
}
```

In [ ]:
#| export
# Literal Definitions from Experimental Constraints 
SourceEnum = Literal["GAIA", "BFCL", "tau-bench"]
ToolCountClassEnum = Literal["single_tool", "multi_tool"]
CompositionDepthEnum = Literal["1", "2_3", "4_plus"]
PlanningHorizonEnum = Literal["short", "medium", "long"]
ScaffoldTypeEnum = Literal["memory", "prompt_scaffold", "multi_agent", "combined"]
GoalConditionTypeEnum = Literal[
    "text_match", 
    "file_creation", 
    "json_state_match", 
    "exit_code", 
    "answer_set_match"
]
KPIEnum = Literal[
    "Reasoning-First Success Rate",
    "Soft-Scaffolding Gain",
    "Fallback Justification Rate",
    "Task Completion Rate",
    "Tool Selection Accuracy",
    "Decomposition Stability",
    "Retry Count per Task",
    "Retry Improvement",
    "Context Degradation",
    "Hallucination Rate",
    "Autonomous Duration",
    "Plan Length vs Failure"
]

# Nested Pydantic Models
class GoalCondition(BaseModel):
    type: GoalConditionTypeEnum
    target: Any  # Can be a string, dict (for json_state), or int (for exit_code)
    match_mode: Literal["exact", "bounded", "includes"]

class InterventionPolicy(BaseModel):
    reasoning_first: bool = True
    soft_scaffold_allowed: List[ScaffoldTypeEnum] = Field(default_factory=list)
    hard_fallback_allowed: bool = False

# Main Baseline Task Schema 
class BaselineTaskSchema(BaseModel):
    """
    Schema for tasks defined in baseline.json
    """
    id: str
    source: SourceEnum
    source_split: str
    task_family: str
    difficulty: str
    prompt: str

    primary_kpi: List[KPIEnum]

    # Required task-shape metadata
    tool_count_class: ToolCountClassEnum
    composition_depth: CompositionDepthEnum
    planning_horizon: PlanningHorizonEnum
    cross_api_mashup: bool

    available_tools_label: List[str]

    goal_condition: GoalCondition
    
    # Preference for "automatic" over human judgment as per design principles
    judge_mode: Literal["automatic", "manual"] = "automatic"
    risk_tags: List[str]

    intervention_policy: InterventionPolicy

We also define a set of tools that are available for the tasks, which are the default `nanobot` tools:

- `cron`: Schedule reminders and recurring tasks.  
- `edit_file`: Edit a file by replacing text.  
- `exec`: Execute shell commands.  
- `glob`: Find files matching a glob pattern.  
- `grep`: Search file contents with regex or plain text.  
- `list_dir`: List directory contents (recursively).  
- `message`: Send messages (including file attachments).  
- `read_file`: Read file contents (with pagination).  
- `spawn`: Spawn a subagent for background tasks.  
- `web_fetch`: Fetch and extract content from URLs.  
- `web_search`: Search the web for titles, URLs, and snippets.  
- `write_file`: Write content to a file.  

In [ ]:
#| export
NANOBOT_TOOLS = [
    "cron",
    "edit_file",
    "exec",
    "glob",
    "grep",
    "list_dir",
    "message",
    "read_file",
    "spawn",
    "web_fetch",
    "web_search",
    "write_file"
]

Save and load tasks to/from JSON files:

In [ ]:
#| export
def save_baseline_tasks(baseline_tasks: List[BaselineTaskSchema], path: Path | str = "data/baseline_tasks.json"):
    path = Path(path)
    if not path.parent.exists():
        path.parent.mkdir(parents=True)
    with open(path, "w") as f:
        json.dump([task.model_dump() for task in baseline_tasks], f, indent=2)

def load_baseline_tasks(path: Path | str = "data/baseline_tasks.json") -> List[BaselineTaskSchema]:
    path = Path(path)
    with open(path, "r") as f:
        data = json.load(f)
    return [BaselineTaskSchema(**item) for item in data]

## GAIA Task Selection

One of our sources of tasks is [GAIA](https://gaia-bench.github.io/), which provides a large collection of real-world multi-step tasks. We will filter the GAIA tasks based on the following criteria:

- **Level**: We will focus on tasks from `level_1` and `level_2`, which are designed to require multi-step reasoning and tool use.  
- **Tools**: We will select tasks that match the default tool set available from `nanobot` 

In [ ]:
#| export
def fetch_gaia():
    print("Fetching GAIA data...")
    # Load the 2023 validation split
    dataset = load_dataset("gaia-benchmark/GAIA", "2023_all", split="validation")
    return dataset

In [ ]:
#| eval: False
# Test it out
gaia_tasks = fetch_gaia()
print(f"Number of tasks: {len(gaia_tasks)}")
print("Sample task:")
print(json.dumps(gaia_tasks[0], indent=2))

Fetching GAIA data...
Number of tasks: 165
Sample task:
{
  "task_id": "c61d22de-5f6c-4958-a7f6-5e9707bd3466",
  "Question": "A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?",
  "Level": "2",
  "Final answer": "egalitarian",
  "file_name": "",
  "file_path": "",
  "Annotator Metadata": {
    "Steps": "1. Go to arxiv.org and navigate to the Advanced Search page.\n2. Enter \"AI regulation\" in the search box and select \"All fields\" from the dropdown.\n3. Enter 2022-06-01 and 2022-07-01 into the date inputs, select \"Submission date (original)\", and submit the search.\n4. Go through the search results to find the article that has a figure with three axes and labels on each end of the axes, titled \"Fairness in Agreement With European Values

In [ ]:
#| eval: false
import re

tools = set()
for task in gaia_tasks:
    if "Annotator Metadata" in task and "Tools" in task["Annotator Metadata"]:
        task_tools = task["Annotator Metadata"]["Tools"].lower().split("\n")
        tools.update([re.sub(r'^\d+\.\s*', '', item) for item in task_tools])
print("Tools used in tasks:")
for tool in tools:
    print(f"- {tool}")

Tools used in tasks:
- a speech-to-text audio processing tool
- code/data analysis tools
- calculator
- google translate access
- computer vision or ocr
- access to google maps
- text editor
- microsoft excel / google sheets
- video processing software
- python compiler
- pdf access
- youtube player
- image recognition/ocr
- access to excel files
- image recognition tools
- a web browser
- spreadsheet editor
- a word reversal tool / script
- a web browser.
- powerpoint viewer
- pdf reader 
- access to the internet archive, web.archive.org
- google search
- web browser
- pdf reader/extracter
- microsoft excel
- video capability
- none
- a calculator.
- calculator (or use excel)
- graph interaction tools
- a search engine.
- wikipedia
- image recognition
- xml file access
- natural language processor
- a speech-to-text tool
- google maps
- word document access
- image recognition and processing tools
- access to youtube
- youtube
- xlsx file access
- audio processing software
- pdf viewe

In [ ]:
#| eval: false
excluded_tools = [
    # Office & Binary Files
    "excel", "spreadsheet", "pdf", "powerpoint", "word document", 
    # Video & Audio
    "video", "youtube", "audio", "speech", 
    # Image & Vision
    "image", "vision", "color", "ocr", "gif", 
    # Highly specific visual/UI apps
    "rubik", "cuniform", "graph interaction", "maps"
]

In [ ]:
#| export
def filter_gaia_tasks(
    tasks: list[dict],  # List of GAIA tasks
    levels: list[int] = [1, 2, 3],  # Difficulty levels to include
    included_tools: list[str] = None,  # Tools that must be included
    excluded_tools: list[str] = None,  # Tools that must not be included
    maximum_steps: int = None,  # Maximum number of steps in the solution
) -> list[dict]:
    """
    Filter GAIA tasks based on specified criteria.
    
    Args:
        tasks: List of GAIA task dictionaries.
        levels: List of difficulty levels to include (default: [1, 2, 3]).
        included_tools: List of tools that must be included in the solution (default: None).
        excluded_tools: List of tools that must not be included in the solution (default: None).
        maximum_steps: Maximum number of steps allowed in the solution (default: None).
    
    Returns:
        A list of filtered GAIA task dictionaries.
    """
    filtered = []
    for task in tasks:
        # Check difficulty level
        if int(task['Level']) not in levels:
            continue
        
        tools = task['Annotator Metadata']['Tools'] \
            if 'Annotator Metadata' in task and 'Tools' in task['Annotator Metadata'] else []
        
        # Check included tools
        if included_tools is not None:
            if not any(tool in tools.lower() for tool in included_tools):
                continue
        
        # Check excluded tools
        if excluded_tools is not None:
            if any(tool in tools.lower() for tool in excluded_tools):
                continue
        
        # Check maximum steps
        if maximum_steps is not None:
            if 'Annotator Metadata' in task and 'Number of steps' in task['Annotator Metadata']:
                if int(task['Annotator Metadata']['Number of steps']) > maximum_steps:
                    continue
        
        filtered.append(task)
    
    return filtered

In [ ]:
#| eval: False
filtered_tasks = filter_gaia_tasks(gaia_tasks, levels=[1, 2], excluded_tools=excluded_tools)
print(f"Number of filtered tasks: {len(filtered_tasks)}")
print("Sample filtered task:")
print(json.dumps(filtered_tasks[0], indent=2))

Number of filtered tasks: 80
Sample filtered task:
{
  "task_id": "17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc",
  "Question": "I\u2019m researching species that became invasive after people who kept them as pets released them. There\u2019s a certain species of fish that was popularized as a pet by being the main character of the movie Finding Nemo. According to the USGS, where was this fish found as a nonnative species, before the year 2020? I need the answer formatted as the five-digit zip codes of the places the species was found, separated by commas if there is more than one place.",
  "Level": "2",
  "Final answer": "34689",
  "file_name": "",
  "file_path": "",
  "Annotator Metadata": {
    "Steps": "1. Search the web for \u201cfinding nemo main character\u201d.\n2. Note the results, which state that the main character is a clownfish.\n3. Search the web for \u201cusgs nonnative species database\u201d.\n4. Click result for the Nonindigenous Aquatic Species site.\n5. Click \u201cMarine Fi

In [ ]:
#| eval: false
tools = set()
for task in filtered_tasks:
    if "Annotator Metadata" in task and "Tools" in task["Annotator Metadata"]:
        task_tools = task["Annotator Metadata"]["Tools"].lower().split("\n")
        tools.update([re.sub(r'^\d+\.\s*', '', item) for item in task_tools])
print("Tools used in tasks:")
for tool in tools:
    print(f"- {tool}")

Tools used in tasks:
- none
- a calculator.
- code/data analysis tools
- calculator or counting function
- access to academic journal websites
- calculator
- text editor
- a search engine.
- markdown
- unlambda compiler (optional)
- a search engine
- no tools required
- wikipedia
- file handling
- computer algebra system
- natural language processor
- search engine
- a web browser
- python
- counter
- a word reversal tool / script
- a calculator
- calculator 
- text processing/diff tool
- a web browser.
- access to the internet archive, web.archive.org
- google search
- web browser
- access to wikipedia


In [ ]:
#| export
def map_gaia_tools(
    tool_name: str,
):
    if not tool_name:
        return None
        
    t = tool_name.lower().strip('. ')
    if t in ["none", "no tools required"]:
        return None
    if t in NANOBOT_TOOLS:
        return t

    # 1. Web Navigation & Fetching (Direct URLs, Wikipedia, Archives)
    if any(keyword in t for keyword in ["browser", "archive", "wikipedia", "journal"]):
        return "web_fetch"

    # 2. Web Searching (Google, generic search engines)
    if "search" in t:
        return "web_search"

    # 3. Code Execution & Computation (Python, calculators, compiling, custom scripts)
    # Since your agent has 'exec', it can use bash/python to do math and text reversal natively.
    if any(keyword in t for keyword in [
        "python", "compiler", "script", "code", "algebra", 
        "calculator", "counter", "counting"
    ]):
        return "exec"

    # 4. File Handling & Editing (Text editors, markdown)
    if any(keyword in t for keyword in ["file handling", "editor", "markdown"]):
        return "edit_file"

    # 5. Text Processing (Diffs, NLP)
    if any(keyword in t for keyword in ["diff", "text processing", "natural language"]):
        return "grep"

    # Fallback: If something slips through, 'exec' is the safest bet for an LLM agent
    # because it allows the agent to figure it out using standard Linux/Python commands.
    return "exec"

In [ ]:
#| export
def map_gaia_row(task: dict) -> BaselineTaskSchema:
    """Translates a raw GAIA row into the strict BaselineTaskSchema."""
    if "Annotator Metadata" not in task:
        print(f"Warning: Task {task.get('task_id', 'unknown')} is missing 'Annotator Metadata'. Defaulting to level 1 assumptions.")
        level = 1
    else:
        level = int(task["Annotator Metadata"].get("Level", 1))
    
    # 1. Level-based Heuristics for the Enums
    if level == 1:
        diff = "easy"
        depth = "1"
        horizon = "short"
        tool_class = "single_tool"
        mashup = False
    elif level == 2:
        diff = "medium"
        depth = "2_3"
        horizon = "medium"
        tool_class = "multi_tool"
        mashup = True
    else:  # Level 3
        diff = "hard"
        depth = "4_plus"
        horizon = "long"
        tool_class = "multi_tool"
        mashup = True

    # 2. Tool Mapping
    if "Annotator Metadata" in task and "Tools" in task["Annotator Metadata"]:
        raw_tools = task["Annotator Metadata"]["Tools"].lower().split("\n")
        tools = [re.sub(r'^\d+\.\s*', '', item) for item in raw_tools]
        mapped_tools = list(set(map_gaia_tools(tool) for tool in tools if map_gaia_tools(tool) is not None))
    else:
        mapped_tools = []

    # 3. Instantiate and Validate
    return BaselineTaskSchema(
        id=task.get("task_id", "unknown"),
        source="GAIA",
        source_split=f"level_{level}",
        
        # GAIA covers many domains, but for a default bucket:
        task_family="general_reasoning", 
        difficulty=diff,
        prompt=task.get("Question", ""),
        
        # Aligned with your KPI-to-benchmark mapping for GAIA
        primary_kpi=["Task Completion Rate", "Decomposition Stability"],
        
        tool_count_class=tool_class,
        composition_depth=depth,
        planning_horizon=horizon,
        cross_api_mashup=mashup,
        available_tools_label=mapped_tools,
        
        goal_condition={
            "type": "text_match",
            "target": str(task.get("Final answer", "")),
            "match_mode": "exact"
        },
        
        judge_mode="automatic",
        risk_tags=["multi_step", "context_fragility"] if level >= 2 else ["single_step"],
        
        intervention_policy={
            "reasoning_first": True,
            "soft_scaffold_allowed": ["prompt_scaffold", "memory", "combined"],
            "hard_fallback_allowed": False
        }
    )

In [ ]:
#| eval: false
baseline_tasks = [map_gaia_row(task) for task in filtered_tasks]
print(f"Number of baseline tasks: {len(baseline_tasks)}")

Number of baseline tasks: 80


In [ ]:
#| eval: false
save_baseline_tasks(baseline_tasks, "../data/gaia_subset.json")

## 